# Notebook 3: Data Preprocessing

## Objective

Merge all race datasets into one master dataset and prepare it for machine learning.


In [2]:
import os
import glob
import pandas as pd
import numpy as np

In [3]:
# finding all the csv files

csv_files = glob.glob("../data/raw/*.csv")
print(f"Found {len(csv_files)} files")
for file in csv_files:
    print(file)

Found 13 files
../data/raw/2024_Spain_laps.csv
../data/raw/2024_Singapore_laps.csv
../data/raw/2023_Spain_laps.csv
../data/raw/2022_Spain_laps.csv
../data/raw/2024_Bahrain_laps.csv
../data/raw/2022_Singapore_laps.csv
../data/raw/2024_Monza_laps.csv
../data/raw/2023_Singapore_laps.csv
../data/raw/2023_Monza_laps.csv
../data/raw/2023_Bahrain_laps.csv
../data/raw/2022_Monza_laps.csv
../data/raw/bahrain_2024_laps.csv
../data/raw/2022_Bahrain_laps.csv


In [4]:
# reading every csv file

dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

# merging everything

master_df = pd.concat(
    dataframes,
    ignore_index=True
)
master_df.head()

,Year,Race,Driver,Team,LapNumber,LapTime,Compound,TyreLife,Stint,Position,TrackStatus
0,2024.0,Spain,VER,Red Bull Racing,1.0,0 days 00:01:23.186000,SOFT,4.0,1.0,2.0,1.0
1,2024.0,Spain,VER,Red Bull Racing,2.0,0 days 00:01:19.871000,SOFT,5.0,1.0,2.0,1.0
2,2024.0,Spain,VER,Red Bull Racing,3.0,0 days 00:01:19.364000,SOFT,6.0,1.0,1.0,1.0
3,2024.0,Spain,VER,Red Bull Racing,4.0,0 days 00:01:20.766000,SOFT,7.0,1.0,1.0,1.0
4,2024.0,Spain,VER,Red Bull Racing,5.0,0 days 00:01:20.827000,SOFT,8.0,1.0,1.0,1.0


In [5]:
# dataset information
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14437 entries, 0 to 14436
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Year         13308 non-null  float64
 1   Race         13308 non-null  object 
 2   Driver       14437 non-null  object 
 3   Team         13308 non-null  object 
 4   LapNumber    14437 non-null  float64
 5   LapTime      14278 non-null  object 
 6   Compound     14437 non-null  object 
 7   TyreLife     14437 non-null  float64
 8   Stint        14437 non-null  float64
 9   Position     14415 non-null  float64
 10  TrackStatus  13308 non-null  float64
dtypes: float64(6), object(5)
memory usage: 1.2+ MB


In [6]:
# printing the shape
print(master_df.shape)

(14437, 11)


In [8]:
# finding the missing values

print(master_df.isnull().sum())

Driver            0
Team           1129
LapNumber         0
LapTime         159
Compound          0
TyreLife          0
Stint             0
Position         22
TrackStatus    1129
dtype: int64


In [8]:
# duplicate row
duplicates = master_df.duplicated().sum()
print("Duplicate rows:", duplicates)

# deleting the duplicates
master_df = master_df.drop_duplicates()
print(master_df.shape)

Duplicate rows: 0
(14437, 11)


In [9]:
# checking the data types

master_df.dtypes

Year           float64
Race            object
Driver          object
Team            object
LapNumber      float64
LapTime         object
Compound        object
TyreLife       float64
Stint          float64
Position       float64
TrackStatus    float64
dtype: object

In [10]:
# converting the lap time and verifying

master_df["LapTime"] = pd.to_timedelta(master_df["LapTime"])
master_df["LapTimeSeconds"] = master_df["LapTime"].dt.total_seconds()

# verifying
master_df[
    ["LapTime", "LapTimeSeconds"]
].head()

,LapTime,LapTimeSeconds
0,0 days 00:01:23.186000,83.186
1,0 days 00:01:19.871000,79.871
2,0 days 00:01:19.364000,79.364
3,0 days 00:01:20.766000,80.766
4,0 days 00:01:20.827000,80.827


In [11]:
# removing the original laptime
master_df.drop(
    columns=["LapTime"],
    inplace=True
)

In [12]:
# saved the processed dataset
os.makedirs("../data/processed", exist_ok=True)
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)
print("Master dataset saved successfully!")

Master dataset saved successfully!


In [13]:
# removing the missing values 

master_df = master_df.dropna(
    subset=[
        "LapTimeSeconds",
        "Position"
    ]
)
print(master_df.shape)

#checking
master_df.isnull().sum()

(14278, 11)


Year              1127
Race              1127
Driver               0
Team              1127
LapNumber            0
Compound             0
TyreLife             0
Stint                0
Position             0
TrackStatus       1127
LapTimeSeconds       0
dtype: int64

In [14]:
# checking the compounds of tire

master_df["Compound"].value_counts()

Compound
HARD            5848
MEDIUM          4033
SOFT            3871
INTERMEDIATE     526
Name: count, dtype: int64

In [15]:
# checking the track status
master_df["TrackStatus"].value_counts().sort_index()


TrackStatus
1.0       12232
2.0          15
4.0         169
6.0           9
12.0        391
14.0         17
16.0          4
21.0         41
24.0         30
26.0         16
41.0         50
64.0          4
67.0          1
71.0          9
124.0         9
126.0        48
164.0        15
216.0         3
671.0        53
712.0         2
1267.0        4
2167.0        1
2671.0       28
Name: count, dtype: int64

In [16]:
# basic stats
master_df.describe()

,Year,LapNumber,TyreLife,Stint,Position,TrackStatus,LapTimeSeconds
count,13151.000000,14278.000000,14278.000000,14278.000000,14278.000000,13151.000000,14278.000000
mean,2023.035587,29.543704,12.585376,2.045244,10.004552,11.579119,94.661850
std,0.816062,17.291230,8.141261,0.886845,5.543455,133.881938,11.944939
min,2022.000000,1.000000,1.000000,1.000000,1.000000,1.000000,76.330000
25%,2022.000000,15.000000,6.000000,1.000000,5.000000,1.000000,86.103250
50%,2023.000000,29.000000,11.000000,2.000000,10.000000,1.000000,96.275500
75%,2024.000000,44.000000,17.000000,3.000000,15.000000,1.000000,99.371750
max,2024.000000,66.000000,50.000000,7.000000,20.000000,2671.000000,149.966000


In [17]:
# saving again
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)

In [18]:
# now we will only keep the dry compounds

master_df = master_df[
    master_df["Compound"].isin([
        "SOFT",
        "MEDIUM",
        "HARD"
    ])
]
print(master_df.shape)

(13752, 11)


In [19]:
# keeping only the green flag laps
master_df = master_df[
    master_df["TrackStatus"] == 1
]
print(master_df.shape)

(11880, 11)


In [20]:
master_df["Compound"].value_counts()

Compound
HARD      4896
MEDIUM    3845
SOFT      3139
Name: count, dtype: int64

In [21]:
master_df["TrackStatus"].value_counts()

TrackStatus
1.0    11880
Name: count, dtype: int64

In [23]:
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)

print(master_df.shape)

(11880, 11)


In [24]:
print(master_df.columns.tolist())
print(master_df.head())

['Year', 'Race', 'Driver', 'Team', 'LapNumber', 'Compound', 'TyreLife', 'Stint', 'Position', 'TrackStatus', 'LapTimeSeconds']
     Year   Race Driver             Team  LapNumber Compound  TyreLife  Stint  \
0  2024.0  Spain    VER  Red Bull Racing        1.0     SOFT       4.0    1.0   
1  2024.0  Spain    VER  Red Bull Racing        2.0     SOFT       5.0    1.0   
2  2024.0  Spain    VER  Red Bull Racing        3.0     SOFT       6.0    1.0   
3  2024.0  Spain    VER  Red Bull Racing        4.0     SOFT       7.0    1.0   
4  2024.0  Spain    VER  Red Bull Racing        5.0     SOFT       8.0    1.0   

   Position  TrackStatus  LapTimeSeconds  
0       2.0          1.0          83.186  
1       2.0          1.0          79.871  
2       1.0          1.0          79.364  
3       1.0          1.0          80.766  
4       1.0          1.0          80.827  
